# Clean Census Data + SFPD District Geography

Run this notebook **top to bottom**.

It loads the team GCP settings from the local `.env`, downloads the newest Census JSON from the team bucket, cleans the Census variables, downloads 2024 Census tract polygons for San Francisco County, joins the cleaned Census rows to tract geometry, and loads the SFPD police district boundary file.

Expected `.env` in the repo root:

```text
GCP_SERVICE_ACCOUNT_KEY=/path/to/git-branch-crime-service-account.json
GCP_PROJECT_ID=git-branch-crime
GCP_BUCKET_NAME=git-branch-crime-bucket
CENSUS_API_KEY=
```


In [1]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt

from dotenv import load_dotenv
from google.oauth2 import service_account
from google.cloud import storage

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "cleaning_eda" else cwd

env_path = project_root / ".env"
load_dotenv(env_path, override=True)

service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")

if not service_account_key:
    raise RuntimeError("GCP_SERVICE_ACCOUNT_KEY is missing from .env")
if not project_id:
    raise RuntimeError("GCP_PROJECT_ID is missing from .env")
if not bucket_name:
    raise RuntimeError("GCP_BUCKET_NAME is missing from .env")

key_path = Path(service_account_key)
if not key_path.is_absolute():
    key_path = project_root / key_path
if not key_path.is_file():
    raise FileNotFoundError(f"Service-account key not found: {key_path}")

credentials = service_account.Credentials.from_service_account_file(str(key_path))
client = storage.Client(project=project_id, credentials=credentials)

print("Project root:", project_root)
print("GCP project:", project_id)
print("Bucket:", bucket_name)
print("Service-account key found:", key_path.is_file())


Project root: /Users/mike/woodbridge/git-branch-crime
GCP project: git-branch-crime
Bucket: git-branch-crime-bucket
Service-account key found: True


In [2]:
census_files = [
    blob for blob in client.list_blobs(bucket_name)
    if blob.name.endswith("_census-data.json")
]
if not census_files:
    raise FileNotFoundError(f"No *_census-data.json files found in bucket {bucket_name}")
latest_file = max(census_files, key=lambda blob: blob.time_created)
census_data = json.loads(latest_file.download_as_text())
census_df = pd.DataFrame(census_data)
print("Census data retrieved successfully!")
print("File:", latest_file.name)
print("Raw dataset shape:", census_df.shape)
display(census_df.head())


Census data retrieved successfully!
File: 2026-10-18_05-58-26_census-data.json
Raw dataset shape: (245, 32)


,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,NAME,B01003_001E,B01002_001E,B19013_001E,B19301_001E,B17001_001E,B17001_002E,B23025_002E,B23025_004E,B23025_005E,...,B15003_025E,B25001_001E,B25003_001E,B25003_002E,B25003_003E,B25064_001E,B25077_001E,state,county,tract
1,Census Tract 101.01; San Francisco County; Cal...,1911,37.6,89022,76257,1911,374,1266,1266,0,...,103,1433,1216,101,1115,2348,1531300,06,075,010101
2,Census Tract 101.02; San Francisco County; Cal...,1917,41.4,102826,82551,1917,155,1039,955,84,...,15,1077,930,264,666,1392,1300000,06,075,010102
3,Census Tract 102.01; San Francisco County; Cal...,2650,33.8,156017,115811,2650,181,1835,1764,23,...,84,1653,1399,406,993,3114,1927600,06,075,010201
4,Census Tract 102.02; San Francisco County; Cal...,1839,38.9,176958,140102,1814,226,1396,1295,101,...,1,1471,1099,299,800,2765,1417300,06,075,010202


In [3]:
census_df.columns = census_df.iloc[0]
census_df = census_df.iloc[1:].reset_index(drop=True)
census_df = census_df.rename(columns={
    "NAME": "census_tract_name",
    "B01003_001E": "total_population",
    "B19013_001E": "median_household_income",
    "B17001_001E": "poverty_status_population",
    "B17001_002E": "population_below_poverty",
    "B23025_004E": "employed_population",
    "B23025_005E": "unemployed_population",
    "B15003_001E": "population_25_and_over",
    "B15003_022E": "bachelors_degree",
    "B15003_023E": "masters_degree",
    "B15003_024E": "professional_degree",
    "B15003_025E": "doctorate_degree",
    "B25003_001E": "occupied_housing_units",
    "B25003_003E": "renter_occupied_units"
})
print("After header fix:", census_df.shape)
display(census_df.head())


After header fix: (244, 32)


,census_tract_name,total_population,B01002_001E,median_household_income,B19301_001E,poverty_status_population,population_below_poverty,B23025_002E,employed_population,unemployed_population,...,doctorate_degree,B25001_001E,occupied_housing_units,B25003_002E,renter_occupied_units,B25064_001E,B25077_001E,state,county,tract
0,Census Tract 101.01; San Francisco County; Cal...,1911,37.6,89022,76257,1911,374,1266,1266,0,...,103,1433,1216,101,1115,2348,1531300,06,075,010101
1,Census Tract 101.02; San Francisco County; Cal...,1917,41.4,102826,82551,1917,155,1039,955,84,...,15,1077,930,264,666,1392,1300000,06,075,010102
2,Census Tract 102.01; San Francisco County; Cal...,2650,33.8,156017,115811,2650,181,1835,1764,23,...,84,1653,1399,406,993,3114,1927600,06,075,010201
3,Census Tract 102.02; San Francisco County; Cal...,1839,38.9,176958,140102,1814,226,1396,1295,101,...,1,1471,1099,299,800,2765,1417300,06,075,010202
4,Census Tract 103; San Francisco County; Califo...,3824,46.2,158144,93972,3824,361,2779,2655,124,...,239,2569,2060,440,1620,2417,1202500,06,075,010300


In [4]:
census_columns = [
    "state", "county", "tract", "total_population",
    "median_household_income", "poverty_status_population",
    "population_below_poverty", "employed_population",
    "unemployed_population", "population_25_and_over",
    "bachelors_degree", "masters_degree", "professional_degree",
    "doctorate_degree", "occupied_housing_units", "renter_occupied_units"
]
census_clean = census_df[census_columns].copy()
census_clean["GEOID"] = (
    census_clean["state"].astype(str).str.zfill(2)
    + census_clean["county"].astype(str).str.zfill(3)
    + census_clean["tract"].astype(str).str.zfill(6)
)
census_clean = census_clean.drop(columns=["state", "county", "tract"])
numeric_columns = census_clean.columns.drop("GEOID")
census_clean[numeric_columns] = census_clean[numeric_columns].apply(pd.to_numeric, errors="coerce")
print("Clean numeric dataset shape:", census_clean.shape)
display(census_clean.head())


Clean numeric dataset shape: (244, 14)


,total_population,median_household_income,poverty_status_population,population_below_poverty,employed_population,unemployed_population,population_25_and_over,bachelors_degree,masters_degree,professional_degree,doctorate_degree,occupied_housing_units,renter_occupied_units,GEOID
0,1911,89022,1911,374,1266,0,1689,843,164,11,103,1216,1115,06075010101
1,1917,102826,1917,155,955,84,1529,382,274,113,15,930,666,06075010102
2,2650,156017,2650,181,1764,23,2240,892,659,82,84,1399,993,06075010201
3,1839,176958,1814,226,1295,101,1567,749,504,115,1,1099,800,06075010202
4,3824,158144,3824,361,2655,124,3360,1526,670,232,239,2060,1620,06075010300


In [5]:
census_clean["poverty_rate"] = (census_clean["population_below_poverty"] / census_clean["poverty_status_population"].replace(0, np.nan)) * 100
census_clean["unemployment_rate"] = (census_clean["unemployed_population"] / (census_clean["employed_population"] + census_clean["unemployed_population"]).replace(0, np.nan)) * 100
census_clean["bachelors_or_higher_rate"] = (census_clean[["bachelors_degree", "masters_degree", "professional_degree", "doctorate_degree"]].sum(axis=1) / census_clean["population_25_and_over"].replace(0, np.nan)) * 100
census_clean["renter_occupancy_rate"] = (census_clean["renter_occupied_units"] / census_clean["occupied_housing_units"].replace(0, np.nan)) * 100
display(census_clean[["GEOID", "poverty_rate", "unemployment_rate", "bachelors_or_higher_rate", "renter_occupancy_rate"]].head())


,GEOID,poverty_rate,unemployment_rate,bachelors_or_higher_rate,renter_occupancy_rate
0,06075010101,19.570905,0.000000,66.370634,91.694079
1,06075010102,8.085550,8.084697,51.275343,71.612903
2,06075010201,6.830189,1.287073,76.651786,70.979271
3,06075010202,12.458655,7.234957,87.364391,72.793449
4,06075010300,9.440377,4.462037,79.375000,78.640777


In [6]:
final_columns = ["GEOID", "total_population", "median_household_income", "poverty_rate", "unemployment_rate", "bachelors_or_higher_rate", "renter_occupancy_rate"]
census_final_df = census_clean[final_columns].copy()
census_final_df.loc[census_final_df["median_household_income"] < 0, "median_household_income"] = np.nan
census_final_df = census_final_df.dropna()
census_final_df = census_final_df[census_final_df["total_population"] > 0].reset_index(drop=True)
print("Final dataset shape:", census_final_df.shape)
print("\nMissing values:")
print(census_final_df.isna().sum())
print("\nDuplicate GEOIDs:", census_final_df["GEOID"].duplicated().sum())
rate_columns = ["poverty_rate", "unemployment_rate", "bachelors_or_higher_rate", "renter_occupancy_rate"]
print("\nRates outside 0–100%:")
print(((census_final_df[rate_columns] < 0) | (census_final_df[rate_columns] > 100)).sum())
display(census_final_df.head())


Final dataset shape: (234, 7)

Missing values:
0
GEOID                       0
total_population            0
median_household_income     0
poverty_rate                0
unemployment_rate           0
bachelors_or_higher_rate    0
renter_occupancy_rate       0
dtype: int64

Duplicate GEOIDs: 0

Rates outside 0–100%:
0
poverty_rate                0
unemployment_rate           0
bachelors_or_higher_rate    0
renter_occupancy_rate       0
dtype: int64


,GEOID,total_population,median_household_income,poverty_rate,unemployment_rate,bachelors_or_higher_rate,renter_occupancy_rate
0,06075010101,1911,89022.0,19.570905,0.000000,66.370634,91.694079
1,06075010102,1917,102826.0,8.085550,8.084697,51.275343,71.612903
2,06075010201,2650,156017.0,6.830189,1.287073,76.651786,70.979271
3,06075010202,1839,176958.0,12.458655,7.234957,87.364391,72.793449
4,06075010300,3824,158144.0,9.440377,4.462037,79.375000,78.640777


In [7]:
tract_url = (
    "https://tigerweb.geo.census.gov/arcgis/rest/services/"
    "TIGERweb/tigerWMS_ACS2024/MapServer/8/query"
    "?where=STATE%3D%2706%27%20AND%20COUNTY%3D%27075%27"
    "&outFields=GEOID"
    "&returnGeometry=true"
    "&f=geojson"
)
tracts_gdf = gpd.read_file(tract_url)
print("Census tract polygons:", tracts_gdf.shape)
print("CRS:", tracts_gdf.crs)
display(tracts_gdf[["GEOID", "geometry"]].head())


Census tract polygons: (244, 2)
CRS: EPSG:4326


,GEOID,geometry
0,06075990100,"POLYGON ((-122.61229 37.81522, -122.60024 37.8..."
1,06075017102,"POLYGON ((-122.45321 37.76825, -122.45335 37.7..."
2,06075012302,"POLYGON ((-122.415 37.78745, -122.41488 37.786..."
3,06075016801,"POLYGON ((-122.43235 37.77378, -122.43247 37.7..."
4,06075011902,"POLYGON ((-122.41246 37.79163, -122.41227 37.7..."


In [8]:
census_geo = census_final_df.merge(tracts_gdf[["GEOID", "geometry"]], on="GEOID", how="inner")
census_geo = gpd.GeoDataFrame(census_geo, geometry="geometry", crs=tracts_gdf.crs)
print("Clean Census tracts with geometry:", census_geo.shape)
print("CRS:", census_geo.crs)
print("Unmatched cleaned GEOIDs:", len(census_final_df) - len(census_geo))
display(census_geo.head())


Clean Census tracts with geometry: (234, 8)
CRS: EPSG:4326
Unmatched cleaned GEOIDs: 0


,GEOID,total_population,median_household_income,poverty_rate,unemployment_rate,bachelors_or_higher_rate,renter_occupancy_rate,geometry
0,06075010101,1911,89022.0,19.570905,0.000000,66.370634,91.694079,"POLYGON ((-122.42108 37.81289, -122.42081 37.8..."
1,06075010102,1917,102826.0,8.085550,8.084697,51.275343,71.612903,"POLYGON ((-122.41689 37.80521, -122.41682 37.8..."
2,06075010201,2650,156017.0,6.830189,1.287073,76.651786,70.979271,"POLYGON ((-122.42484 37.80227, -122.42482 37.8..."
3,06075010202,1839,176958.0,12.458655,7.234957,87.364391,72.793449,"POLYGON ((-122.42685 37.80902, -122.42683 37.8..."
4,06075010300,3824,158144.0,9.440377,4.462037,79.375000,78.640777,"POLYGON ((-122.41872 37.80593, -122.4186 37.80..."


In [9]:
police_path = project_root / "Current_Police_Districts_20261017.csv"
police_df = pd.read_csv(police_path)
police_df["geometry"] = police_df["the_geom"].apply(wkt.loads)
police_gdf = gpd.GeoDataFrame(police_df, geometry="geometry", crs="EPSG:4326")
print("Police district polygons:", police_gdf.shape)
print("Districts:", sorted(police_gdf["district"].unique()))
display(police_gdf[["district", "geometry"]].head())


Police district polygons: (10, 9)
Districts: ['BAYVIEW', 'CENTRAL', 'INGLESIDE', 'MISSION', 'NORTHERN', 'PARK', 'RICHMOND', 'SOUTHERN', 'TARAVAL', 'TENDERLOIN']


,district,geometry
0,MISSION,"MULTIPOLYGON (((-122.41755 37.77003, -122.4175..."
1,TENDERLOIN,"MULTIPOLYGON (((-122.4086 37.78381, -122.40864..."
2,NORTHERN,"MULTIPOLYGON (((-122.42534 37.80421, -122.4249..."
3,PARK,"MULTIPOLYGON (((-122.42647 37.76974, -122.4266..."
4,RICHMOND,"MULTIPOLYGON (((-122.44041 37.78719, -122.4404..."


In [10]:
# Assign each Census tract to an SFPD police district
# using the tract centroid.

census_centroids = census_geo.copy()

# Use a projected CRS before calculating centroids
census_centroids = census_centroids.to_crs("EPSG:26910")
police_projected = police_gdf.to_crs("EPSG:26910")

census_centroids["geometry"] = census_centroids.geometry.centroid

census_district = gpd.sjoin(
    census_centroids,
    police_projected[["district", "geometry"]],
    how="left",
    predicate="within"
)

census_district = census_district.rename(
    columns={"district": "police_district"}
)

print("Census tracts:", len(census_district))
print("Matched to police district:",
      census_district["police_district"].notna().sum())
print("Unmatched:",
      census_district["police_district"].isna().sum())

print("\nTracts by police district:")
print(census_district["police_district"].value_counts().sort_index())

Census tracts: 234
Matched to police district: 228
Unmatched: 6

Tracts by police district:
police_district
BAYVIEW       19
CENTRAL       25
INGLESIDE     30
MISSION       26
NORTHERN      30
PARK          15
RICHMOND      22
SOUTHERN      18
TARAVAL       33
TENDERLOIN    10
Name: count, dtype: int64


In [11]:
unmatched_tracts = census_district[
    census_district["police_district"].isna()
][
    [
        "GEOID",
        "total_population",
        "median_household_income"
    ]
]

unmatched_tracts

,GEOID,total_population,median_household_income
3,06075010202,1839,176958.0
7,06075010500,3473,179896.0
37,06075012602,3071,244000.0
190,06075035201,5020,136295.0
210,06075047904,3077,105577.0
211,06075060100,3901,219750.0


In [ ]:
unmatched = census_district[
    census_district["police_district"].isna()
].copy()

unmatched = unmatched.drop(
    columns=["index_right", "police_district"],
    errors="ignore"
)

unmatched_nearest = gpd.sjoin_nearest(
    unmatched,
    police_projected[["district", "geometry"]],
    how="left",
    distance_col="distance_to_district"
)

unmatched_nearest = unmatched_nearest.rename(
    columns={"district": "police_district"}
)

unmatched_nearest[
    ["GEOID", "police_district", "distance_to_district"]
]

,GEOID,police_district,distance_to_district
3,06075010202,CENTRAL,13.517963
7,06075010500,CENTRAL,21.260119
37,06075012602,NORTHERN,4.839544
190,06075035201,TARAVAL,14.705785
210,06075047904,RICHMOND,1018.773685
211,06075060100,RICHMOND,2213.606249


In [15]:
# Keep the 228 direct matches
matched = census_district[
    census_district["police_district"].notna()
].copy()

matched = matched.drop(columns=["index_right"], errors="ignore")

# Accept only the 4 very-close boundary cases
close_matches = unmatched_nearest[
    unmatched_nearest["distance_to_district"] <= 50
].copy()

# Keep the 2 distant tracts unresolved
far_unmatched = unmatched_nearest[
    unmatched_nearest["distance_to_district"] > 50
].copy()

# Combine accepted tract assignments
census_district_final = pd.concat(
    [matched, close_matches],
    ignore_index=True
)

print("Final assigned tracts:", len(census_district_final))
print("Still unresolved:", len(far_unmatched))

print("\nAssigned tracts by police district:")
print(
    census_district_final["police_district"]
    .value_counts()
    .sort_index()
)

print("\nStill unresolved:")
print(
    far_unmatched[
        ["GEOID", "police_district", "distance_to_district"]
    ]
)

Final assigned tracts: 232
Still unresolved: 2

Assigned tracts by police district:
police_district
BAYVIEW       19
CENTRAL       27
INGLESIDE     30
MISSION       26
NORTHERN      31
PARK          15
RICHMOND      22
SOUTHERN      18
TARAVAL       34
TENDERLOIN    10
Name: count, dtype: int64

Still unresolved:
           GEOID police_district  distance_to_district
210  06075047904        RICHMOND           1018.773685
211  06075060100        RICHMOND           2213.606249


In [16]:
# Use polygon overlap for the 6 centroid-unmatched tracts

unmatched_geoids = unmatched_nearest["GEOID"].tolist()

unmatched_polygons = census_geo[
    census_geo["GEOID"].isin(unmatched_geoids)
].copy()

# Project to meters for valid area calculations
unmatched_polygons = unmatched_polygons.to_crs("EPSG:26910")
police_overlap = police_gdf.to_crs("EPSG:26910")

overlap = gpd.overlay(
    unmatched_polygons,
    police_overlap[["district", "geometry"]],
    how="intersection"
)

overlap["overlap_area"] = overlap.geometry.area

largest_overlap = (
    overlap.sort_values("overlap_area", ascending=False)
    .drop_duplicates("GEOID")
    [["GEOID", "district", "overlap_area"]]
    .sort_values("GEOID")
)

largest_overlap

,GEOID,district,overlap_area
1,06075010202,CENTRAL,332916.185708
2,06075010500,CENTRAL,683563.377373
4,06075012602,NORTHERN,347499.661491
6,06075035201,TARAVAL,695551.644608
7,06075047904,RICHMOND,336509.886077
9,06075060100,RICHMOND,67600.689525


In [17]:
# Final Census tract -> police district assignments

direct_matches = census_district[
    census_district["police_district"].notna()
].copy()

direct_matches = direct_matches.drop(
    columns=["index_right"],
    errors="ignore"
)

overlap_assignments = (
    unmatched_polygons[
        [
            "GEOID",
            "total_population",
            "median_household_income",
            "poverty_rate",
            "unemployment_rate",
            "bachelors_or_higher_rate",
            "renter_occupancy_rate",
            "geometry"
        ]
    ]
    .merge(
        largest_overlap[["GEOID", "district"]],
        on="GEOID",
        how="left"
    )
    .rename(columns={"district": "police_district"})
)

census_district_final = pd.concat(
    [direct_matches, overlap_assignments],
    ignore_index=True
)

print("Final assigned tracts:", len(census_district_final))
print("Unassigned:", census_district_final["police_district"].isna().sum())

print("\nTracts by police district:")
print(
    census_district_final["police_district"]
    .value_counts()
    .sort_index()
)

Final assigned tracts: 234
Unassigned: 0

Tracts by police district:
police_district
BAYVIEW       19
CENTRAL       27
INGLESIDE     30
MISSION       26
NORTHERN      31
PARK          15
RICHMOND      24
SOUTHERN      18
TARAVAL       34
TENDERLOIN    10
Name: count, dtype: int64


In [18]:
# Aggregate Census data to SFPD police district

# Bring the original Census count columns back in
census_with_counts = census_district_final.merge(
    census_clean[
        [
            "GEOID",
            "poverty_status_population",
            "population_below_poverty",
            "employed_population",
            "unemployed_population",
            "population_25_and_over",
            "bachelors_degree",
            "masters_degree",
            "professional_degree",
            "doctorate_degree",
            "occupied_housing_units",
            "renter_occupied_units"
        ]
    ],
    on="GEOID",
    how="left"
)

district_census = (
    census_with_counts
    .groupby("police_district", as_index=False)
    .agg(
        total_population=("total_population", "sum"),
        poverty_status_population=("poverty_status_population", "sum"),
        population_below_poverty=("population_below_poverty", "sum"),
        employed_population=("employed_population", "sum"),
        unemployed_population=("unemployed_population", "sum"),
        population_25_and_over=("population_25_and_over", "sum"),
        bachelors_degree=("bachelors_degree", "sum"),
        masters_degree=("masters_degree", "sum"),
        professional_degree=("professional_degree", "sum"),
        doctorate_degree=("doctorate_degree", "sum"),
        occupied_housing_units=("occupied_housing_units", "sum"),
        renter_occupied_units=("renter_occupied_units", "sum")
    )
)

# Recompute district-level rates
district_census["poverty_rate"] = (
    district_census["population_below_poverty"]
    / district_census["poverty_status_population"]
) * 100

district_census["unemployment_rate"] = (
    district_census["unemployed_population"]
    / (
        district_census["employed_population"]
        + district_census["unemployed_population"]
    )
) * 100

district_census["bachelors_or_higher_rate"] = (
    district_census[
        [
            "bachelors_degree",
            "masters_degree",
            "professional_degree",
            "doctorate_degree"
        ]
    ].sum(axis=1)
    / district_census["population_25_and_over"]
) * 100

district_census["renter_occupancy_rate"] = (
    district_census["renter_occupied_units"]
    / district_census["occupied_housing_units"]
) * 100

print("District rows:", len(district_census))

display(
    district_census[
        [
            "police_district",
            "total_population",
            "poverty_rate",
            "unemployment_rate",
            "bachelors_or_higher_rate",
            "renter_occupancy_rate"
        ]
    ]
)

District rows: 10


,police_district,total_population,poverty_rate,unemployment_rate,bachelors_or_higher_rate,renter_occupancy_rate
0,BAYVIEW,68095,12.890475,7.945434,43.817028,47.649301
1,CENTRAL,64182,14.324515,6.137055,59.210938,83.334265
2,INGLESIDE,127269,8.649632,6.044934,48.071594,36.102317
3,MISSION,83586,9.602497,5.851620,67.881652,63.256034
4,NORTHERN,92370,11.954413,4.970952,72.794137,76.647129
5,PARK,58215,10.080436,4.665145,77.273986,65.366278
6,RICHMOND,84792,6.546203,4.863641,70.334871,62.497019
7,SOUTHERN,61357,14.647221,6.834833,68.894316,71.704482
8,TARAVAL,149225,8.438796,5.911970,57.043937,39.219037
9,TENDERLOIN,26945,27.278239,11.445179,34.708790,95.056934


In [19]:
# Approximate district median household income
# using a population-weighted average of tract medians

income_by_district = (
    census_district_final
    .assign(
        weighted_income=lambda x:
            x["median_household_income"] * x["total_population"]
    )
    .groupby("police_district", as_index=False)
    .agg(
        weighted_income_sum=("weighted_income", "sum"),
        population_sum=("total_population", "sum")
    )
)

income_by_district["median_household_income_approx"] = (
    income_by_district["weighted_income_sum"]
    / income_by_district["population_sum"]
)

district_census = district_census.merge(
    income_by_district[
        ["police_district", "median_household_income_approx"]
    ],
    on="police_district",
    how="left"
)

display(
    district_census[
        [
            "police_district",
            "total_population",
            "median_household_income_approx",
            "poverty_rate",
            "unemployment_rate",
            "bachelors_or_higher_rate",
            "renter_occupancy_rate"
        ]
    ]
)

,police_district,total_population,median_household_income_approx,poverty_rate,unemployment_rate,bachelors_or_higher_rate,renter_occupancy_rate
0,BAYVIEW,68095,134621.281651,12.890475,7.945434,43.817028,47.649301
1,CENTRAL,64182,112001.237465,14.324515,6.137055,59.210938,83.334265
2,INGLESIDE,127269,147412.967808,8.649632,6.044934,48.071594,36.102317
3,MISSION,83586,174355.651114,9.602497,5.851620,67.881652,63.256034
4,NORTHERN,92370,156761.044008,11.954413,4.970952,72.794137,76.647129
5,PARK,58215,184048.077746,10.080436,4.665145,77.273986,65.366278
6,RICHMOND,84792,157220.962756,6.546203,4.863641,70.334871,62.497019
7,SOUTHERN,61357,165507.995730,14.647221,6.834833,68.894316,71.704482
8,TARAVAL,149225,151874.993714,8.438796,5.911970,57.043937,39.219037
9,TENDERLOIN,26945,51029.541510,27.278239,11.445179,34.708790,95.056934


In [20]:
final_census_district = district_census[
    [
        "police_district",
        "total_population",
        "median_household_income_approx",
        "poverty_rate",
        "unemployment_rate",
        "bachelors_or_higher_rate",
        "renter_occupancy_rate"
    ]
].copy()

output_path = project_root / "cleaning_eda" / "census_by_police_district.csv"

final_census_district.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", final_census_district.shape)

Saved: /Users/mike/woodbridge/git-branch-crime/cleaning_eda/census_by_police_district.csv
Shape: (10, 7)
